# Prompt Engineering Lab

**學習目標：**
- [ ] Zero-shot vs. Few-shot vs. CoT 對科學文本抽取的影響
- [ ] Self-consistency 如何提高數值抽取的可靠性
- [ ] Structured output (JSON mode) 的使用

**預計時間：** 3-4 小時

> 使用 Anthropic API + `claude-haiku-4-5`（便宜、支援 temperature、200K context）。`.env` 中要設定 `ANTHROPIC_API_KEY`。

In [ ]:
import os, json, sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')
print('API key set:', bool(os.getenv('ANTHROPIC_API_KEY')))

## 1. 測試文本

用同一段文本比較不同 prompting 策略的效果

In [ ]:
SAMPLE_TEXT = """
The perovskite oxide La₀.₈Sr₀.₂MnO₃ (LSMO) thin films were deposited via pulsed laser
deposition (PLD) at a substrate temperature of 700°C under an oxygen partial pressure
of 200 mTorr. Film thickness: approximately 50 nm. Post-deposition annealing at 800°C
for 2 hours. Metal-insulator transition at T_MI = 370 K.
"""
print(SAMPLE_TEXT)

## 2. Zero-shot vs. CoT 比較

In [ ]:
from chain_of_thought import zero_shot_extraction, zero_shot_cot_extraction, few_shot_cot_extraction
import anthropic
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env

# Zero-shot
result_zero = zero_shot_extraction(SAMPLE_TEXT, client)
print('Zero-shot:')
print(json.dumps(result_zero, indent=2, ensure_ascii=False))

In [ ]:
# Zero-shot CoT
result_cot = zero_shot_cot_extraction(SAMPLE_TEXT, client)
print('Zero-shot CoT:')
print(json.dumps(result_cot, indent=2, ensure_ascii=False))

In [ ]:
# Few-shot CoT
result_fs = few_shot_cot_extraction(SAMPLE_TEXT, client)
print('Few-shot CoT:')
print(json.dumps(result_fs, indent=2, ensure_ascii=False))

## 3. Self-consistency (N=5 samples)

多路徑推理 + 多數決，觀察 confidence 分數。低 confidence 的欄位 = 未來 fine-tuning 的 hard cases。

In [ ]:
from self_consistency import self_consistent_extraction
result_sc = self_consistent_extraction(SAMPLE_TEXT, client, n_samples=5)

## 4. ReAct pattern（Reasoning + Acting）

看 agent 如何交替 `Thought → Action → Observation`。這是 Week 3 LangGraph agent 的雛形。

In [ ]:
from react_pattern import react_agent

question = """From this text, extract all experimental parameters and convert
oxygen pressure to Pa and substrate temperature to K:

'LSMO thin films were deposited via PLD at 700°C substrate temperature under
200 mTorr O₂. Film thickness was ~50 nm.'

Return as JSON."""

react_agent(question, client)

## 練習題

1. 修改 SAMPLE_TEXT 為一段你自己找到的材料科學論文摘要，比較三種方法的效果
2. 新增一個含有科學記號（如 5×10⁻⁷）的文本，觀察哪種方法處理得更好
3. 把 `MODEL = "claude-haiku-4-5"` 改成 `"claude-sonnet-4-6"`，看更強的模型差多少
4. 設計 domain-specific 的 few-shot 範例（LSMO、鈣鈦礦、ZnO...），看 few-shot 範例的領域相關性如何影響結果


In [ ]:
# 你的實驗
MY_TEXT = """
# TODO: 貼上你的論文段落
"""

# result = zero_shot_cot_extraction(MY_TEXT, client)
# print(json.dumps(result, indent=2, ensure_ascii=False))